# week09 · 01 후처리 체인 — 폴더 하나 → 표 한 줄
덱 2 · 5 · 11 · 17 · 22장. 순서: run_md prod → density → ced_method_ab → viscosity_gk → run_md tg → tg_fit → results.csv.
LAMMPS 가 있으면 ①→③ 전부(≈ 5분 · 1코어) · 없으면 `--skip-md` 로 expected_output 의 로그를 복사해 후처리만.

실행(기본): 내 컴퓨터의 mdai · week09 폴더에서 Claude Code → 덱 26장 P9-3 지시(같은 체인을 `scripts/` 로 직접 실행 · `../prompts.md`). 이 노트북 실행 = week09 폴더에서 `python ../run_nb.py notebooks/01_postprocess.ipynb`(작업 폴더 = week09 · nbconvert 는 경로가 어긋나 쓰지 않음) · 결과는 JupyterLab 으로 열어 본다. Colab 은 선택 경로.

In [ ]:
# ① 저장소 받기 (처음 1회 · Colab 등 저장소가 없을 때만) · week09 로 이동 · 로컬 mdai 이면 이미 week09 폴더
import sys
if "google.colab" in sys.modules:   # (선택) Colab 일 때만 · 로컬(Claude Code · run_nb.py)은 주차 폴더에서 바로 실행
    !git clone https://github.com/sentirain7/2026_MOOC_Lecture_Materials.git 2>/dev/null
    %cd */week09
# 기본 = 로컬 mdai(LAMMPS 포함 · environment.yml) · Claude Code. Colab(선택)에서 LAMMPS 가 필요하면 (수 분): !pip -q install condacolab && import condacolab; condacolab.install()  → 런타임 재시작 뒤  !mamba install -y -c conda-forge lammps


In [ ]:
# ② 체인 실행 — LAMMPS 가 있으면 전부 · 없으면 expected_output 로그로 후처리만
import shutil, pathlib, subprocess, sys
HAVE_LMP = shutil.which('lmp') is not None
if not HAVE_LMP:
    pathlib.Path('out/prod').mkdir(parents=True, exist_ok=True); pathlib.Path('out/tg').mkdir(parents=True, exist_ok=True)
    for f in ['log.prod','stress.txt','nvt_final.data']: shutil.copy(f'expected_output/prod/{f}', f'out/prod/{f}')
    for f in pathlib.Path('expected_output/tg').glob('tg_T*.txt'): shutil.copy(f, 'out/tg/'+f.name)
args = [sys.executable, 'scripts/postprocess_all.py', '--exp-id', 'toluene50_w09'] + ([] if HAVE_LMP else ['--skip-md'])
print(subprocess.run(args, text=True, capture_output=True).stdout[-1500:])


In [ ]:
# ③ 결과 읽기 — results.csv 한 줄 · JSON 4
import json, csv
print(open('out/results.csv').read())
for n in ['density','ced','viscosity_gk','tg']:
    d = json.load(open(f'out/{n}.json')); print(n, {k: d[k] for k in list(d)[:6]})


In [ ]:
# ④ 그림 2장 — 밀도 누적 평균(덱 5장) · GK 러닝 적분(덱 17장)
import matplotlib.pyplot as plt, numpy as np
s = np.loadtxt('out/density_series.csv', delimiter=',', skiprows=1); g = np.loadtxt('out/gk_curve.csv', delimiter=',', skiprows=1)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(s[:,1], s[:,2], color='#B9AE9E', lw=1, label='순간값'); ax[0].plot(s[:,1], s[:,3], color='#C8610C', lw=2.5, label='누적 평균'); ax[0].set_xlabel('t (ps)'); ax[0].set_ylabel('ρ (g/cm³)'); ax[0].legend()
ax[1].plot(g[:,0], g[:,2], color='#2E8B57', lw=2.5); ax[1].set_xlabel('t (fs)'); ax[1].set_ylabel('η(t) (mPa·s)'); ax[1].set_title('러닝 적분 — 평탄 전 (20 ps)')
plt.tight_layout(); plt.savefig('out/w09_density_gk.png', dpi=120); plt.show()
